# Hands-on AI Agent untuk Pemula
## Daily Sync & Team Dispatcher — memakai Google Sheets Project Tracker

**Use case:** agent membaca project tracker yang sudah digunakan tim, merangkum progres, menyoroti tugas berisiko, lalu membuat draf email daily sync untuk direview.

### Yang akan dipelajari
1. Mengakses Google Sheets dari Google Colab dengan autentikasi Google.
2. Memahami tools dan function calling.
3. Membaca kolom `Task ID`, `Project`, `Department`, `Task / Deliverable`, `Owner`, `Priority`, `Status`, `Progress (%)`, `Start Date`, `Due Date`, `Days to Due`, `Overdue?`, `Last Update`, dan `Blocker / Notes`.
4. Membuat agent yang menggunakan data aktual, bukan data dummy.
5. Membuat draf koordinasi tanpa mengirim email otomatis.

**Keamanan:** notebook akan meminta izin Google saat dijalankan. Pastikan akun yang dipakai memiliki akses ke spreadsheet. Jangan membagikan token atau API key. Draf email pada latihan ini tetap disimpan lokal dan tidak dikirim.

## 0. Arsitektur

`Google Sheets → Tool baca tracker → AI Agent → Tool buat draf → Review manusia`

**Catatan akses:** notebook ini memakai Google Colab authentication untuk membaca spreadsheet. Link yang diberikan menunjukkan spreadsheet ID `1ziqSNHWRiqst3vgXP-ZSdTOS4SgtZ_TS`. Jika spreadsheet tidak dapat diakses, periksa apakah akun Google yang dipilih memiliki permission Viewer atau lebih tinggi.

## 1. Instalasi library

In [1]:
%pip -q install -U openai pandas gspread

## 2. Autentikasi Google Colab dan baca Google Sheets

In [2]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd

creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_ID = "1Wn1QVN5PmjminBDSnJtEnuKmvqSZSUpMGAMczu-e2ME"
spreadsheet = gc.open_by_key(SPREADSHEET_ID)

print("Spreadsheet berhasil dibuka:", spreadsheet.title)
print("Worksheet tersedia:", [ws.title for ws in spreadsheet.worksheets()])

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Spreadsheet berhasil dibuka: Project Management
Worksheet tersedia: ['Project Tracker']


### Pilih worksheet yang berisi project tracker

Cell berikut memakai worksheet pertama sebagai default. Jika data berada di tab lain, ubah `WORKSHEET_NAME` sesuai nama tab yang terlihat pada output sebelumnya.

In [3]:
WORKSHEET_NAME = spreadsheet.sheet1.title  # ubah jika tracker berada di tab lain
worksheet = spreadsheet.worksheet(WORKSHEET_NAME)

records = worksheet.get_all_records()
tasks = pd.DataFrame(records)

print("Worksheet:", WORKSHEET_NAME)
print("Jumlah task:", len(tasks))
display(tasks.head())

Worksheet: Project Tracker
Jumlah task: 300


,Task ID,Project,Department,Task / Deliverable,Owner,Priority,Status,Progress (%),Start Date,Due Date,Days to Due,Overdue?,Last Update,Blocker / Notes
0,TASK-001,Sales Performance Analytics,Data & Analytics,Mengumpulkan kebutuhan data,Budi Santoso,Medium,In Progress,79%,05-Sep-2026,23-Nov-2026,24,No,24-Okt-2026,Koordinasi lintas tim berlangsung
1,TASK-002,PMO Transformation,Project Management,Menyusun project charter,Citra Amelia,Critical,In Progress,35%,28-Agu-2026,03-Nov-2026,4,No,27-Okt-2026,Sedang dikerjakan
2,TASK-003,Support Process Improvement,Customer Success,Memantau SLA respons,Rizky Maulana,Medium,Completed,100%,01-Agu-2026,04-Nov-2026,5,No,25-Okt-2026,Selesai sesuai rencana
3,TASK-004,Data Quality Initiative,Data & Analytics,Membersihkan dataset operasional,Kevin Hartono,Medium,Completed,100%,13-Agu-2026,20-Nov-2026,21,No,29-Okt-2026,Dokumentasi telah diperbarui
4,TASK-005,Service Delivery Improvement,Operations,Memeriksa checklist operasional,Aldi Wijaya,Low,In Review,91%,07-Sep-2026,15-Nov-2026,16,No,29-Okt-2026,Menunggu review stakeholder


## 3. Validasi kolom dan tipe data

Kolom pada tracker bisa saja sedikit berbeda penulisannya. Cell ini menampilkan nama kolom aktual dan memeriksa kolom minimum yang dibutuhkan agent.

In [4]:
print("Kolom yang ditemukan:")
for col in tasks.columns:
    print("-", repr(col))

REQUIRED_COLUMNS = [
    "Task ID", "Project", "Department", "Task / Deliverable", "Owner",
    "Priority", "Status", "Progress (%)", "Start Date", "Due Date",
    "Days to Due", "Overdue?", "Last Update", "Blocker / Notes"
]

missing = [c for c in REQUIRED_COLUMNS if c not in tasks.columns]
if missing:
    print("\nPerlu perhatian: kolom berikut tidak ditemukan persis seperti yang diharapkan:")
    print(missing)
    print("Periksa nama header di Google Sheets dan sesuaikan REQUIRED_COLUMNS atau pemetaan kolom.")
else:
    print("\nSemua kolom utama ditemukan.")

Kolom yang ditemukan:
- 'Task ID'
- 'Project'
- 'Department'
- 'Task / Deliverable'
- 'Owner'
- 'Priority'
- 'Status'
- 'Progress (%)'
- 'Start Date'
- 'Due Date'
- 'Days to Due'
- 'Overdue?'
- 'Last Update'
- 'Blocker / Notes'

Semua kolom utama ditemukan.


## 4. Kenali data: tugas berisiko dan deadline

Jangan langsung meminta LLM menyimpulkan semuanya. Gunakan pandas untuk memeriksa data terstruktur terlebih dahulu. Ini juga membantu agent tidak mengandalkan teks saja.

In [5]:
# Bersihkan nilai teks untuk kolom penting
for col in ["Priority", "Status", "Owner", "Project", "Overdue?"]:
    if col in tasks.columns:
        tasks[col] = tasks[col].fillna("").astype(str).str.strip()

if "Progress (%)" in tasks.columns:
    tasks["Progress (%)"] = pd.to_numeric(
        tasks["Progress (%)"].astype(str).str.replace("%", "", regex=False),
        errors="coerce"
    )

display(tasks[["Task ID", "Project", "Task / Deliverable", "Owner", "Priority",
               "Status", "Progress (%)", "Due Date", "Days to Due", "Overdue?", "Blocker / Notes"]].head())

,Task ID,Project,Task / Deliverable,Owner,Priority,Status,Progress (%),Due Date,Days to Due,Overdue?,Blocker / Notes
0,TASK-001,Sales Performance Analytics,Mengumpulkan kebutuhan data,Budi Santoso,Medium,In Progress,79,23-Nov-2026,24,No,Koordinasi lintas tim berlangsung
1,TASK-002,PMO Transformation,Menyusun project charter,Citra Amelia,Critical,In Progress,35,03-Nov-2026,4,No,Sedang dikerjakan
2,TASK-003,Support Process Improvement,Memantau SLA respons,Rizky Maulana,Medium,Completed,100,04-Nov-2026,5,No,Selesai sesuai rencana
3,TASK-004,Data Quality Initiative,Membersihkan dataset operasional,Kevin Hartono,Medium,Completed,100,20-Nov-2026,21,No,Dokumentasi telah diperbarui
4,TASK-005,Service Delivery Improvement,Memeriksa checklist operasional,Aldi Wijaya,Low,In Review,91,15-Nov-2026,16,No,Menunggu review stakeholder


## 5. Buat Tool 1 — membaca project tracker

In [6]:
from typing import Optional

def read_project_tracker(
    status: Optional[str] = None,
    owner: Optional[str] = None,
    priority: Optional[str] = None,
    project: Optional[str] = None,
    overdue_only: bool = False
):
    """Baca dan filter task dari Google Sheets yang sudah dimuat ke DataFrame."""
    df = tasks.copy()

    filters = [
        ("Status", status),
        ("Owner", owner),
        ("Priority", priority),
        ("Project", project),
    ]
    for col, value in filters:
        if value:
            if col not in df.columns:
                raise ValueError(f"Kolom '{col}' tidak ditemukan.")
            df = df[df[col].astype(str).str.casefold() == str(value).casefold()]

    if overdue_only:
        if "Overdue?" not in df.columns:
            raise ValueError("Kolom 'Overdue?' tidak ditemukan.")
        df = df[df["Overdue?"].astype(str).str.casefold().isin(["yes", "ya", "true", "1"])]

    # Hilangkan NaN agar hasil aman dikirim sebagai JSON ke model
    df = df.where(pd.notna(df), None)
    return df.to_dict(orient="records")

# Uji tool langsung
print("Total task:", len(read_project_tracker()))
print("\nTask berstatus In Progress:", len(read_project_tracker(status="In Progress")))
print("\nTask overdue:", len(read_project_tracker(overdue_only=True)))

Total task: 300

Task berstatus In Progress: 127

Task overdue: 44


## 6. Buat Tool 2 — draf email (simulasi aman)

Untuk latihan pemula, tool ini **belum membuat draf Gmail sungguhan**. Ia menyimpan draf dalam notebook agar kita bisa menguji agent tanpa mengirim pesan keluar. Integrasi Gmail API dapat ditambahkan setelah alur dan persetujuan pengguna siap.

In [7]:
email_drafts = []

def create_email_draft(to: str, subject: str, body: str):
    """Membuat draf email lokal untuk direview. Tool ini tidak mengirim email."""
    draft = {
        "to": to,
        "subject": subject,
        "body": body,
        "status": "DRAFT_ONLY — belum dikirim"
    }
    email_drafts.append(draft)
    return {
        "ok": True,
        "message": "Draf disimpan di notebook; tidak ada email yang dikirim.",
        "draft": draft
    }

## 7. Siapkan OpenAI API

In [8]:
import os
from google import genai
from google.colab import userdata

# Mengambil API key dari Secrets Manager Colab dengan nama 'GOOGLE_API_KEY'
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    from getpass import getpass
    GOOGLE_API_KEY = getpass("Masukkan GOOGLE_API_KEY Anda: ")

client = genai.Client(api_key=GOOGLE_API_KEY)

## 8. Definisikan tools untuk function calling

In [9]:
tools = [
    {
        "type": "function",
        "name": "read_project_tracker",
        "description": "Baca task dari Google Sheets project tracker. Gunakan sebelum membuat ringkasan. Bisa filter berdasarkan status, owner, priority, project, atau overdue.",
        "parameters": {
            "type": "object",
            "properties": {
                "status": {"type": ["string", "null"], "description": "Filter Status, misalnya In Progress, Completed, atau In Review."},
                "owner": {"type": ["string", "null"], "description": "Nama owner."},
                "priority": {"type": ["string", "null"], "description": "Filter Priority, misalnya Critical, High, Medium, Low."},
                "project": {"type": ["string", "null"], "description": "Nama project."},
                "overdue_only": {"type": "boolean", "description": "True jika hanya ingin task overdue."}
            },
            "required": ["status", "owner", "priority", "project", "overdue_only"],
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "create_email_draft",
        "description": "Membuat draf email lokal untuk direview. Tool ini tidak mengirim email.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"],
            "additionalProperties": False
        },
        "strict": True
    }
]
print("Tools siap:", [t["name"] for t in tools])

Tools siap: ['read_project_tracker', 'create_email_draft']


## 9. Bangun agent loop

In [12]:
import json
from google.genai import types

def execute_tool(name, args):
    print(f"[Agent] Menjalankan tool '{name}' dengan argumen: {args}")
    if name == "read_project_tracker":
        return read_project_tracker(**args)
    if name == "create_email_draft":
        return create_email_draft(**args)
    raise ValueError(f"Tool tidak diizinkan: {name}")

def run_agent(user_request, team_email="team@example.com", max_tool_rounds=6):
    instructions = f"""
Kamu adalah Daily Sync & Team Dispatcher untuk tim profesional.
Sumber kebenaran status pekerjaan adalah Google Sheets yang dibaca lewat read_project_tracker.
- Baca tracker sebelum membuat kesimpulan.
- Gunakan kolom Task ID, Project, Department, Task / Deliverable, Owner, Priority, Status,
  Progress (%), Due Date, Days to Due, Overdue?, Last Update, dan Blocker / Notes bila tersedia.
- Prioritaskan Critical/High, task overdue, task yang statusnya Blocked, dan deadline terdekat.
- PENTING: Saat menyebutkan atau mendaftar pekerjaan, selalu tulis nomor task (Task ID) bersamaan dengan nama tugasnya (Task / Deliverable) dengan format `[Task ID] Nama Tugas` (Contoh: [TASK-001] Mengumpulkan kebutuhan data).
- Jangan mengarang fakta. Jika data kosong atau tidak konsisten, sebutkan keterbatasannya.
- Bedakan blocker yang tercatat dari risiko yang kamu simpulkan.
- Buat ringkasan, blocker, dan action items dengan owner dan langkah berikutnya bila dapat ditentukan.
- Tulis bahasa Indonesia yang ramah, ringkas, profesional.
- Simpan hasil sebagai draf menggunakan create_email_draft, penerima: {team_email}.
- Jangan pernah mengklaim email telah dikirim.
"""

    # Memetakan tools format otomatis menggunakan definisi fungsi Python langsung
    gemini_tools = [read_project_tracker, create_email_draft]

    config = types.GenerateContentConfig(
        system_instruction=instructions,
        tools=gemini_tools,
        temperature=0.2
    )

    chat = client.chats.create(model="gemini-3-flash-preview", config=config)
    response = chat.send_message(user_request)

    for round_num in range(max_tool_rounds):
        if not response.function_calls:
            break

        tool_responses = []
        for call in response.function_calls:
            # Bersihkan argumen dari tipe data yang tidak sesuai jika ada
            args = {k: v for k, v in call.args.items() if v is not None}
            try:
                result = execute_tool(call.name, args)
            except Exception as e:
                print(f"[Error] Gagal menjalankan {call.name}: {e}")
                result = {"error": str(e)}

            tool_responses.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )

        # Kirim balik hasil eksekusi tool ke model untuk putaran berikutnya
        response = chat.send_message(tool_responses)

    return response.text

## 10. Jalankan Daily Sync Agent

In [13]:
request = """
Siapkan daily sync berdasarkan project tracker terbaru.
Buat ringkasan eksekutif singkat, soroti task Critical/High, task overdue atau mendekati deadline,
status yang berpotensi menghambat, dan action items per owner. Simpan sebagai draf email.
"""

result = run_agent(request, team_email="faransina.rumere@gmail.com")
print(result)

Draf email untuk daily sync telah berhasil disiapkan dan disimpan. Berikut adalah ringkasan dari laporan tersebut:

### **Ringkasan Daily Sync**
Laporan ini menyoroti kondisi proyek yang memiliki cukup banyak tugas **Overdue** dan **Blocked**. Fokus utama tim hari ini adalah melakukan akselerasi pada tugas-tugas kritis dan menyelesaikan hambatan koordinasi.

### **Poin Utama:**
*   **Tugas Kritis & Overdue:** Perhatian khusus diberikan pada `[TASK-033] Memperbarui dokumentasi teknis` (Rizky Maulana) dan `[TASK-279] Melakukan deployment ke production` (Kevin Hartono) yang saat ini terhambat.
*   **Hambatan (Blockers):** Mayoritas hambatan disebabkan oleh ketergantungan pada data eksternal/vendor serta menunggu persetujuan stakeholder (misalnya pada TASK-078, TASK-118, dan TASK-160).
*   **Deadline Terdekat:** Beberapa tugas seperti `[TASK-081]`, `[TASK-146]`, dan `[TASK-202]` memiliki deadline hari ini atau besok.

### **Action Items Utama:**
*   **Kevin Hartono & Andi Pratama:** Fokus 

## 11. Review draf — tidak ada email terkirim

In [14]:
for i, draft in enumerate(email_drafts, start=1):
    print(f"===== DRAF {i} =====")
    print("To:", draft["to"])
    print("Subject:", draft["subject"])
    print("Status:", draft["status"])
    print("\n" + draft["body"] + "\n")

===== DRAF 1 =====
To: faransina.rumere@gmail.com
Subject: Daily Sync & Team Dispatch - Project Status Update
Status: DRAFT_ONLY — belum dikirim

Halo Tim,

Berikut adalah ringkasan status proyek dan rencana kerja berdasarkan project tracker terbaru untuk koordinasi harian kita.

### Ringkasan Eksekutif
Saat ini terdapat sejumlah besar tugas yang berstatus Overdue (terlambat) dan Blocked (terhambat). Fokus utama hari ini adalah penyelesaian backlog prioritas tinggi dan koordinasi lintas tim untuk membuka hambatan pada tugas-tugas kritis.

### Sorotan Prioritas (Critical/High)
- **[TASK-033] Memperbarui dokumentasi teknis** (Rizky Maulana) - Status: Not Started (Critical & Overdue)
- **[TASK-279] Melakukan deployment ke production** (Kevin Hartono) - Status: Blocked (Critical & Overdue)
- **[TASK-002] Menyusun project charter** (Citra Amelia) - Progress: 35% (Critical)
- **[TASK-013] Melakukan rekonsiliasi biaya** (Rafi Hidayat) - Progress: 43% (Critical)
- **[TASK-079] Memproses reimbu

## 12. Latihan untuk peserta

Coba setiap prompt berikut dan bandingkan hasilnya:
1. “Tampilkan hanya task Critical.”
2. “Ringkas pekerjaan yang menjadi tanggung jawab Budi Santoso.”
3. “Fokus hanya pada task overdue.”
4. “Siapkan daily sync per project.”
5. “Nyatakan semua pekerjaan sudah selesai.” — apakah agent menolak klaim yang bertentangan dengan tracker?

### Tantangan lanjutan
- Tambahkan filter Department.
- Tambahkan pemeriksaan data kosong dan duplikat Task ID.
- Hitung sendiri task yang overdue dari Due Date dan tanggal saat ini, lalu bandingkan dengan kolom `Overdue?`.
- Buat format pesan singkat untuk chat tim.
- Tambahkan tes yang memeriksa apakah draf menyertakan owner dan blocker yang benar.

## 13. Mengubah simulasi menjadi Gmail Draft sungguhan

Setelah latihan berfungsi, ganti isi `create_email_draft()` dengan integrasi Gmail API `users.drafts.create` melalui OAuth. Gunakan scope Gmail yang sesuai dan uji pada akun latihan terlebih dahulu. Jangan memakai endpoint pengiriman (`send`) pada versi pemula ini.

Untuk penggunaan produksi, pertimbangkan:
- autentikasi dan akses minimal;
- logging pemanggilan tool;
- penanganan error API;
- validasi penerima dan isi;
- konfirmasi manusia sebelum pesan dikirim;
- refresh data tracker saat agent dijalankan, bukan memakai snapshot yang sudah lama.

## 14. Ringkasan konsep

- **LLM:** memahami instruksi dan menyusun bahasa.
- **Tool:** fungsi yang memberi akses ke data atau tindakan.
- **Function calling:** mekanisme model meminta aplikasi menjalankan tool tertentu.
- **Agent loop:** aplikasi menjalankan tool, mengembalikan hasil, dan membiarkan model melanjutkan.
- **Human-in-the-loop:** manusia tetap memeriksa draf sebelum komunikasi keluar.